# Post-Training Step 3: RLHF with GenRM

**Paper §3.3** — after RLVR, RLHF is applied to improve behaviour on chat-style
tasks where rewards are harder to verify automatically.
A **Generative Reward Model (GenRM)** scores responses.

### Key design choices
| Feature | Detail |
|---|---|
| Scoring | Circular pairwise comparison → O(N) GenRM calls |
| Length control | Group Relative Length Control (GRLC, §3.3.2 Eq. 6) |
| Conciseness bonus | Quality-Gated Conciseness Bonus for shortest high-quality response |
| Update | Same GRPO as RLVR, KL anchored to RLVR checkpoint |

### Notebook contents
1. Environment setup
2. Imports & hyperparameters
3. Tokenizer
4. Dataset
5. Model (load RLVR checkpoint)
6. GenRM helpers
7. GRLC helpers
8. Training loop


## 0. Environment Setup

Detects Colab / Kaggle, installs packages, and adds the repo to `sys.path`.

In [ ]:
import os, sys

IN_COLAB  = False
IN_KAGGLE = os.path.exists("/kaggle/working") and os.path.exists("/kaggle/input")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}")


In [ ]:
if IN_COLAB or IN_KAGGLE:
    # Set ACCELERATOR to "cuda12" (GPU) or "tpu" (Colab TPU only).
    ACCELERATOR = "cuda12"
    import subprocess
    subprocess.run(
        ["pip", "install", "-q",
         f"jax[{ACCELERATOR}]", "flax", "optax", "orbax-checkpoint",
         "datasets", "transformers", "matplotlib"],
        check=True,
    )


In [ ]:
import pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-jax-nemotron.git"


def _detect_repo_root() -> pathlib.Path:
    """Detect local repo root by searching upward for key project files."""
    cwd = pathlib.Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "training_shared.py").exists() and (candidate / "nemotron.py").exists():
            return candidate
    return cwd


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
elif IN_KAGGLE:
    REPO_DIR = pathlib.Path("/kaggle/working/nugie-jax-nemotron")
    if not REPO_DIR.exists():
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    REPO_DIR = _detect_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("REPO_DIR:", REPO_DIR)
print(f"NUM_DEVICES={NUM_DEVICES}")


## 1. Imports & Hyperparameters

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
from transformers import AutoTokenizer

from moe import SparseMoE
from nemotron import NemotronConfig, NemotronNanoBlock
from training_shared import (
    RLVR_CKPT_DIR, RLHF_CKPT_DIR, RL_TRAIN_SEQ_LEN,
    RLHF_NUM_PROMPTS, RLHF_NUM_RESPONSES,
    RLHF_LAMBDA_THINK, RLHF_LAMBDA_ANSWER,
    RLHF_BETA_THINK, RLHF_BETA_ANSWER, RLHF_PERCENTILE,
    RLHF_LR, RLHF_WD, RLHF_B1, RLHF_B2,
    RLHF_STEPS, RLHF_CKPT_EVERY,
    build_model, collect_moe_layers, make_constant_lr_optimizer,
    load_sft_data, generate_completion_tokens,
    compute_grpo_advantages, build_grpo_batch,
    rl_step, compute_log_probs, update_moe_biases,
    make_checkpoint_manager, save_checkpoint, try_load_from_dir,
    NUM_DEVICES,
)

print("JAX devices:", jax.devices())


In [ ]:
TRAIN_STEPS   = RLHF_STEPS          # 50
NUM_PROMPTS   = RLHF_NUM_PROMPTS    # 4
NUM_RESPONSES = RLHF_NUM_RESPONSES  # 4
CKPT_DIR      = RLHF_CKPT_DIR

print(f"RLHF: STEPS={TRAIN_STEPS} | prompts/step={NUM_PROMPTS} | responses/prompt={NUM_RESPONSES}")
print(f"NUM_DEVICES={NUM_DEVICES}")


## 2. Tokenizer

In [ ]:
print("Loading Nemotron tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained("nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {tokenizer.vocab_size}")


## 3. Dataset

In [ ]:
train_samples = load_sft_data("train")

# Cache prompt token IDs/text once to avoid re-tokenising each step.
rlhf_prompt_ids: list[list[int]] = []
rlhf_prompt_texts: list[str] = []
for sample in train_samples:
    prompt_text = f"User: {sample['question']}\nAssistant: "
    rlhf_prompt_texts.append(prompt_text)
    rlhf_prompt_ids.append(tokenizer.encode(prompt_text, add_special_tokens=False))

print(f"Train samples : {len(train_samples)}")


## 4. Model — Load RLVR Checkpoint

In [ ]:
print("Building model ...")
model = build_model(seed=0)
moe_layers = collect_moe_layers(model)

loaded = try_load_from_dir(RLVR_CKPT_DIR, model, model.config)
if loaded:
    print("Resumed from RLVR checkpoint.")
else:
    print("No RLVR checkpoint found — starting from random init.")

# Frozen copy of the RLVR checkpoint as reference policy for KL penalty.
graphdef, ref_state = nnx.split(model)
ref_model = nnx.merge(graphdef, ref_state)


## 5. Optimizer

In [ ]:
tx        = make_constant_lr_optimizer(RLHF_LR, RLHF_WD, RLHF_B1, RLHF_B2)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)
ckpt_mgr  = make_checkpoint_manager(CKPT_DIR)
print(f"Constant-LR AdamW (lr={RLHF_LR}).")


## 6. GenRM Scoring

**Paper §3.3.1** — the real GenRM is a large LLM (Qwen3-235B) that evaluates
pairwise response quality via circular comparison.  Below we use a heuristic proxy.

In [ ]:
def simulated_genrm_score(response_text: str, prompt: str) -> float:
    """Simulate a GenRM helpfulness score in [1, 5]."""
    score = 1.0
    if "<think>"  in response_text: score += 1.0
    if "</think>" in response_text: score += 1.0
    if len(response_text.split()) > 10: score += 1.0
    return min(score, 5.0)


def circular_pairwise_comparison(responses: list[str], prompt: str) -> list[float]:
    """Score N responses in O(N) circular comparisons (§3.3.2)."""
    N = len(responses)
    accumulated = np.zeros(N, dtype=np.float32)
    for i in range(N):
        j = (i + 1) % N
        s_i = simulated_genrm_score(responses[i], prompt)
        s_j = simulated_genrm_score(responses[j], prompt)
        accumulated[i] += s_i
        accumulated[j] += s_j
    return (accumulated / 2.0).tolist()


## 7. Group Relative Length Control (GRLC)

**Paper §3.3.2, Eq. 6** — zero-sum length penalty within the response group;
shorter responses are rewarded when quality exceeds the threshold.

In [ ]:
def apply_group_relative_length_control(
    base_scores: list[float],
    responses: list[str],
) -> list[float]:
    """Adjust scores with GRLC and Quality-Gated Conciseness Bonus."""
    N = len(responses)
    if N == 0:
        return base_scores

    think_lens, answer_lens = [], []
    for r in responses:
        if "</think>" in r:
            think_part  = r.split("</think>")[0]
            answer_part = r.split("</think>")[1]
        else:
            think_part  = ""; answer_part = r
        think_lens. append(len(think_part.split()))
        answer_lens.append(len(answer_part.split()))

    think_arr  = np.array(think_lens,  dtype=np.float32)
    answer_arr = np.array(answer_lens, dtype=np.float32)

    def centered_weights(lengths: np.ndarray) -> np.ndarray:
        lo, hi = lengths.min(), lengths.max()
        if hi == lo:
            return np.zeros_like(lengths)
        w = 1.0 - (lengths - lo) / (hi - lo)
        return w - w.mean()

    scores  = np.array(base_scores, dtype=np.float32)
    scores += RLHF_LAMBDA_THINK  * centered_weights(think_arr)
    scores += RLHF_LAMBDA_ANSWER * centered_weights(answer_arr)

    threshold      = float(np.percentile(base_scores, RLHF_PERCENTILE))
    min_think_idx  = int(np.argmin(think_arr))
    min_answer_idx = int(np.argmin(answer_arr))

    if base_scores[min_think_idx]  >= threshold: scores[min_think_idx]  += RLHF_BETA_THINK
    if base_scores[min_answer_idx] >= threshold: scores[min_answer_idx] += RLHF_BETA_ANSWER

    return scores.tolist()


## 8. Training Loop

In [ ]:
print(f"\n=== Post-Training Step 3: RLHF with GenRM ===")
print(f"Training for {TRAIN_STEPS} steps ...\n")

score_history: list[float] = []

for step in range(TRAIN_STEPS):
    start = step * NUM_PROMPTS
    end = (step + 1) * NUM_PROMPTS
    batch_indices = list(range(start, min(end, len(train_samples))))
    if not batch_indices:
        break

    prompt_ids_list:   list[list[int]]       = []
    completion_groups: list[list[list[int]]] = []
    reward_groups:     list[list[float]]     = []

    for sample_idx in batch_indices:
        prompt_text = rlhf_prompt_texts[sample_idx]
        p_ids = rlhf_prompt_ids[sample_idx]
        prompt_ids_list.append(p_ids)

        responses_text: list[str]       = []
        completions:    list[list[int]] = []
        for g in range(NUM_RESPONSES):
            comp_ids  = generate_completion_tokens(
                model, tokenizer, p_ids, max_new_tokens=200, rng_seed=step * 50 + g
            )
            comp_text = tokenizer.decode(comp_ids, skip_special_tokens=True)
            responses_text.append(comp_text)
            completions.append(comp_ids)

        base_scores     = circular_pairwise_comparison(responses_text, prompt_text)
        adjusted_scores = apply_group_relative_length_control(base_scores, responses_text)

        completion_groups.append(completions)
        reward_groups.append(adjusted_scores)

    advantage_groups = [compute_grpo_advantages(rg) for rg in reward_groups]
    token_ids, masks, advantages = build_grpo_batch(
        prompt_ids_list, completion_groups, advantage_groups,
        seq_len=RL_TRAIN_SEQ_LEN, pad_id=tokenizer.eos_token_id,
    )
    ref_log_probs = compute_log_probs(ref_model, token_ids)
    old_log_probs = compute_log_probs(model, token_ids)
    loss = rl_step(model, optimizer, token_ids, masks, advantages, ref_log_probs, old_log_probs)
    update_moe_biases(moe_layers)

    mean_score = float(np.mean([s for rg in reward_groups for s in rg]))
    score_history.append(mean_score)

    if step % 10 == 0:
        print(f"  RLHF step {step:3d} | loss={float(loss):.4f} | "
              f"mean_genrm_score={mean_score:.3f}")

    if step % RLHF_CKPT_EVERY == 0:
        save_checkpoint(ckpt_mgr, model, step)

save_checkpoint(ckpt_mgr, model, TRAIN_STEPS)
print("\nRLHF complete.")


## 9. GenRM Score Curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, len(score_history) + 1), score_history)
    plt.xlabel("Step"); plt.ylabel("Mean GenRM Score"); plt.title("RLHF Mean GenRM Score per Step")
    plt.grid(True); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — skipping plot.")
